In [1]:
import pandas as pd
import numpy as np
import datetime
import os
#from typing import List, Tuple, Any, Callable
from itertools import accumulate

from dorieh.cms.fts2yaml import mcr_type, MedicareFTS
# from dorieh.platform.loader.data_loader import DataLoader
# from dorieh.cms.mcr_data_loader import MedicareDataLoader
# from dorieh.cms.tools.mcr_fts2db import MedicareLoader
# from dorieh.utils.fwf import FWFReader
# from dorieh.utils.io_utils import fopen

In [2]:
# fast line parser for fixed-width files
def fwf_parser(fieldwidths):
    # get absolute cut points for columns
    cuts = tuple(cut for cut in accumulate(abs(fw) for fw in fieldwidths))
    # turn into tuples
    flds = tuple(zip((0,)+cuts, cuts))
    # format into string and pass to eval() for lazy evaluation
    slcs = ', '.join(f'line[{i}:{j}]' for i, j in flds)
    parse = eval('lambda line: ({})\n'.format(slcs))  # Create and compile source code.
    return parse

# takes in a df and a dict of types (supported: ["NUM", "CHAR", "DATE"]) and casts columns into those types
def change_dftypes(df, type_dict, verbose=False):
    for col in df.columns:
        if type_dict[col] == 'NUM':
            if verbose: 
                print(col + ": CHAR --> NUM")
            df[col] = pd.to_numeric(df[col].str.strip(), errors='coerce')
        elif type_dict[col] == 'DATE':
            if verbose: 
                print(col + ": CHAR --> DATE")
            df[col] = pd.to_datetime(df['COVSTART'], errors='coerce')
        else:
            df[col] = df[col].str.strip().replace('', np.nan)
            if verbose: 
                print(col + ": CHAR")
    return(df)

In [3]:
# loading basic information about the .dat file
fts_path = "/n/dominici_nsaph_l3/Lab/data/ci3_d_medicare/original_data/cms_medicare/data/4334/2012/mbsf_ab_summary_res000017155_req004334_2012.fts"

f, _ = os.path.splitext(fts_path)
basedir, fname = os.path.split(f)
t = mcr_type(fname)
dat_path = f + ".dat"
parquet_path = f + ".parquet"

# getting column names
fts = MedicareFTS(t).init(fts_path) # init FTS object
fts_meta = fts.to_fwf_meta(dat_path) # get metadata
colnm_lst = [col.name for col in fts_meta.columns] # names of cols
colwidth_lst = [col.length for col in fts_meta.columns] # widths of cols
type_dict = {col.name: col.type for col in fts_meta.columns} # types of cols


In [4]:
parse = fwf_parser(colwidth_lst) # initialize parser
row_max = 1000000 # process 10^6 rows for this example

# read raw fixed width file
start_read = datetime.datetime.now()
print("Reading raw fw file " + dat_path + "...")
with open(dat_path, "r") as f:
    lines = f.read().split("\n")
    f.close()
print("Time to read " + str(len(lines)) + " rows: " + str(datetime.datetime.now() - start_read))

if row_max > len(lines):
    row_max = len(lines)

# split rows into columns according to the column width
start_parse = datetime.datetime.now()
prs_df_tmp = pd.DataFrame([parse(line) for line in lines[:row_max]], 
                            columns=colnm_lst)
print("Time to parse " + str(row_max) + " rows: " + str(datetime.datetime.now() - start_parse))


processed_df = prs_df_tmp.copy() # making a copy just for logging purposes right now
# cast columns into different datatypes
start_cast = datetime.datetime.now()
processed_df = change_dftypes(processed_df, type_dict)
print("Time to cast " + str(row_max) + " rows: " + str(datetime.datetime.now() - start_parse))

Reading raw fw file /n/dominici_nsaph_l3/Lab/data/ci3_d_medicare/original_data/cms_medicare/data/4334/2012/mbsf_ab_summary_res000017155_req004334_2012.dat...
Time to read 53597184 rows: 0:00:49.125899
Time to parse 1000000 rows: 0:00:10.378697


/tmp/ipykernel_3101856/3694625630.py:24: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[col] = df[col].str.strip().replace('', np.nan)


Time to cast 1000000 rows: 0:00:26.093889
